# EV Purchase — GPU Tri-Model Ensemble Blend
LightGBM + CatBoost + XGBoost GPU blend with 5-fold stratified stacking

`GPU · ROC-AUC · Ensemble`

---

In high-dimensional tabular classification problems ($N \approx 668\text{K}$ rows), single gradient boosted trees often plateau. Maximizing the Receiver Operating Characteristic Area Under the Curve (ROC-AUC) requires ensembling diverse model families:

> ⚡ **Why Multi-Model Ensembling Works:**
> Different boosting algorithms partition feature space differently:
> - **LightGBM:** Leaf-wise (best-first) tree growth with gradient-based one-side sampling (GOSS).
> - **CatBoost:** Symmetric (oblivious) trees providing natural regularization and fast GPU tensor inference.
> - **XGBoost:** Exact/histogram depth-wise splitting with second-order Taylor expansion gradients.

> 📐 **Optimal Convex Blending Formulation:**
> We solve for the optimal weight vector $\mathbf{w}^* = [w_{\text{LGB}}, w_{\text{CAT}}, w_{\text{XGB}}]^T$ that maximizes the out-of-fold ROC-AUC:
> $$
> \mathbf{w}^* = \arg\max_{\mathbf{w}} \text{AUC}\left(y, \sum_{m=1}^{M} w_m \hat{y}_m \right) \quad \text{s.t.} \quad \sum_{m=1}^{M} w_m = 1, \; w_m \ge 0
> $$

### 🗺️ Methodology Architecture
1. **GPU Acceleration Diagnostics:** CUDA hardware validation.
2. **Advanced Tabular Feature Engineering:** 21 composite domain interactions.
3. **Stratified 5-Fold Cross-Validation:** Zero-leakage out-of-fold probability caching.
4. **Tri-Model Training (LGBM + CatBoost GPU + XGBoost GPU):** Fast parallel gradient boosting.
5. **Convex Ensemble Blending (SLSQP Optimization):** Mathematically optimal probability weighting.
6. **Leaderboard Submission Generation:** Generating verified continuous probabilities.

In [ ]:
# ── Core Libraries, GPU Detection & Styling ──
import os
import gc
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.optimize import minimize

# Gradient Boosting Ecosystem
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Check GPU Hardware
cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else 'CPU Only'
print(f"🚀 Hardware Acceleration: CUDA Available={cuda_available} | Device: {gpu_name}")

# TokyoNight Visual Aesthetic
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#414868'
plt.rcParams['axes.linewidth'] = 1.2

COLOR_CYAN = '#7aa2f7'
COLOR_PURPLE = '#bb9af7'
COLOR_ORANGE = '#ff9e64'
COLOR_GREEN = '#9ece6a'
COLOR_RED = '#f7768e'
COLOR_YELLOW = '#e0af68'
BG_SURFACE = '#1a1b26'
print("✓ TokyoNight theme & modeling libraries initialized.")


## 📥 2. Data Ingestion & Advanced Feature Engineering Pipeline
We engineer composite domain features that capture affordability, commute intensity, and infrastructure accessibility.

In [ ]:
comp_dir = '/kaggle/input/playground-series-s6e9'
if not os.path.exists(comp_dir):
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    comp_dir = os.path.dirname(matches[0]) if matches else '.'

train_raw = pd.read_csv(os.path.join(comp_dir, 'train.csv'))
test_raw = pd.read_csv(os.path.join(comp_dir, 'test.csv'))

print(f"📊 Training Set: {train_raw.shape[0]:,} rows | Test Set: {test_raw.shape[0]:,} rows")

# Target Extraction
if train_raw['Will_Buy_EV'].dtype == object:
    y = (train_raw['Will_Buy_EV'].str.strip().str.lower() == 'yes').astype(int).values
else:
    y = train_raw['Will_Buy_EV'].astype(int).values

def engineer_features(df):
    data = df.copy()
    
    # 1. Infrastructure Access Ratios
    home_cs = data['Charging_Stations_Near_Home'].fillna(0)
    work_cs = data['Charging_Stations_Near_Work'].fillna(0)
    data['Total_Charging_Access'] = home_cs + work_cs
    data['Charging_Home_Ratio'] = (home_cs + 0.1) / (work_cs + home_cs + 0.2)
    
    # 2. Economic Fleet Capacity & Normalized Affordability
    cars = data['Number_of_Cars_Owned'].fillna(1).clip(lower=1)
    data['Income_Per_Vehicle'] = data['Annual_Income_USD'] / cars
    data['Log_Income'] = np.log1p(data['Annual_Income_USD'].clip(lower=0))
    data['Income_To_Age'] = data['Annual_Income_USD'] / (data['Age'].clip(lower=18))
    
    # 3. Commute Friction & Range Anxiety Mapping
    anxiety_map = {'Low': 0, 'Medium': 1, 'High': 2}
    data['Range_Anxiety_Num'] = data['Range_Anxiety_Level'].map(anxiety_map).fillna(1)
    data['Commute_Anxiety_Friction'] = data['Daily_Commute_km'] * (data['Range_Anxiety_Num'] + 1)
    data['Commute_Per_Car'] = data['Daily_Commute_km'] / cars
    
    # 4. Binary Convenience Flags
    data['Home_Charge_Bin'] = (data['Home_Charging_Possible'].astype(str).str.lower() == 'yes').astype(int)
    data['Subsidy_Bin'] = (data['Subsidy_Available'].astype(str).str.lower() == 'yes').astype(int)
    
    # 5. Composite Adoption Synergy
    env_score = data['Environmental_Concern_Level'].fillna(3.0)
    data['Adoption_Synergy'] = (data['Home_Charge_Bin'] * 1.5) + (data['Subsidy_Bin'] * 1.0) + (env_score * 0.5)
    
    # 6. Categorical Encodings
    cat_cols = ['Gender', 'City_Type', 'Current_Car_Type']
    for col in cat_cols:
        data[col] = data[col].astype('category').cat.codes
        
    return data

train_fe = engineer_features(train_raw)
test_fe = engineer_features(test_raw)

feature_cols = [
    'Age', 'Annual_Income_USD', 'Log_Income', 'Income_To_Age', 'Daily_Commute_km',
    'Number_of_Cars_Owned', 'Income_Per_Vehicle', 'Commute_Per_Car',
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Total_Charging_Access', 'Charging_Home_Ratio',
    'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type',
    'Home_Charge_Bin', 'Subsidy_Bin', 'Range_Anxiety_Num',
    'Commute_Anxiety_Friction', 'Adoption_Synergy'
]

X = train_fe[feature_cols].fillna(-1)
X_test = test_fe[feature_cols].fillna(-1)
print(f"✓ Engineered {len(feature_cols)} structured features for training.")


## 🚀 3. Stratified 5-Fold Training: LightGBM, CatBoost (GPU) & XGBoost (GPU)
We train all three diverse tree families across identical Stratified 5-Fold splits to maintain strict zero-leakage out-of-fold alignment.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# OOF and Test Prediction Containers
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

print("Training 3-Model GPU Stratified 5-Fold Pipeline...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n▶ FOLD {fold+1} / 5")
    X_tr, y_tr = X.iloc[trn_idx], y[trn_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]
    
    # 1. LightGBM
    model_lgb = LGBMClassifier(
        n_estimators=380,
        learning_rate=0.06,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(X_tr, y_tr)
    p_va_lgb = model_lgb.predict_proba(X_va)[:, 1]
    oof_lgb[val_idx] = p_va_lgb
    test_lgb += model_lgb.predict_proba(X_test)[:, 1] / 5.0
    print(f"  • LightGBM Fold {fold+1} AUC: {roc_auc_score(y_va, p_va_lgb):.5f}")
    
    # 2. CatBoost (GPU / CPU adaptive)
    cat_task = 'GPU' if cuda_available else 'CPU'
    model_cat = CatBoostClassifier(
        iterations=400,
        learning_rate=0.07,
        depth=6,
        task_type=cat_task,
        eval_metric='AUC',
        random_seed=42 + fold,
        verbose=0
    )
    model_cat.fit(X_tr, y_tr)
    p_va_cat = model_cat.predict_proba(X_va)[:, 1]
    oof_cat[val_idx] = p_va_cat
    test_cat += model_cat.predict_proba(X_test)[:, 1] / 5.0
    print(f"  • CatBoost Fold {fold+1} AUC: {roc_auc_score(y_va, p_va_cat):.5f}")
    
    # 3. XGBoost (GPU Hist / CPU adaptive)
    xgb_tree = 'hist'
    xgb_dev = 'cuda' if cuda_available else 'cpu'
    model_xgb = XGBClassifier(
        n_estimators=380,
        learning_rate=0.06,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method=xgb_tree,
        device=xgb_dev,
        random_state=42 + fold,
        eval_metric='auc'
    )
    model_xgb.fit(X_tr, y_tr)
    p_va_xgb = model_xgb.predict_proba(X_va)[:, 1]
    oof_xgb[val_idx] = p_va_xgb
    test_xgb += model_xgb.predict_proba(X_test)[:, 1] / 5.0
    print(f"  • XGBoost Fold {fold+1} AUC:  {roc_auc_score(y_va, p_va_xgb):.5f}")

print("\n🏆 Individual Out-Of-Fold Baseline Scores:")
print(f"  • LightGBM OOF ROC-AUC: {roc_auc_score(y, oof_lgb):.5f}")
print(f"  • CatBoost OOF ROC-AUC: {roc_auc_score(y, oof_cat):.5f}")
print(f"  • XGBoost  OOF ROC-AUC: {roc_auc_score(y, oof_xgb):.5f}")


## ⚖️ 4. Convex Ensemble Optimization (SLSQP Blending)
We use **Sequential Least Squares Programming (SLSQP)** to find optimal non-negative weights that strictly maximize the OOF ROC-AUC.

In [ ]:
preds_matrix = np.column_stack([oof_lgb, oof_cat, oof_xgb])
test_matrix = np.column_stack([test_lgb, test_cat, test_xgb])

def loss_func(weights):
    w = weights / np.sum(weights)
    blend = np.dot(preds_matrix, w)
    return -roc_auc_score(y, blend)

init_weights = [0.333, 0.333, 0.334]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = {'type': 'eq', 'fun': lambda w: 1 - sum(w)}

res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = res.x / np.sum(res.x)

oof_blend = np.dot(preds_matrix, opt_weights)
test_blend = np.dot(test_matrix, opt_weights)

best_auc = roc_auc_score(y, oof_blend)
print(f"🎯 Optimal Blending Weights:")
print(f"   • LightGBM: {opt_weights[0]:.3f}")
print(f"   • CatBoost: {opt_weights[1]:.3f}")
print(f"   • XGBoost:  {opt_weights[2]:.3f}")
print(f"\n🔥 BLENDED ENSEMBLE OOF ROC-AUC: {best_auc:.5f} (vs Single Best: {max(roc_auc_score(y, oof_lgb), roc_auc_score(y, oof_cat), roc_auc_score(y, oof_xgb)):.5f})")


## 📈 5. Diagnostic ROC Curves & Ensemble Gain
We inspect the Receiver Operating Characteristic curves comparing individual models to the blended ensemble.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG_SURFACE)
ax.set_facecolor(BG_SURFACE)

# LightGBM ROC
fpr_lgb, tpr_lgb, _ = roc_curve(y, oof_lgb)
ax.plot(fpr_lgb, tpr_lgb, color=COLOR_CYAN, alpha=0.6, label=f'LightGBM (AUC = {roc_auc_score(y, oof_lgb):.4f})')

# CatBoost ROC
fpr_cat, tpr_cat, _ = roc_curve(y, oof_cat)
ax.plot(fpr_cat, tpr_cat, color=COLOR_PURPLE, alpha=0.6, label=f'CatBoost (AUC = {roc_auc_score(y, oof_cat):.4f})')

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y, oof_xgb)
ax.plot(fpr_xgb, tpr_xgb, color=COLOR_GREEN, alpha=0.6, label=f'XGBoost (AUC = {roc_auc_score(y, oof_xgb):.4f})')

# Optimal Ensemble ROC
fpr_bl, tpr_bl, _ = roc_curve(y, oof_blend)
ax.plot(fpr_bl, tpr_bl, color=COLOR_ORANGE, linewidth=2.5, label=f'★ Blended Ensemble (AUC = {best_auc:.4f})')

# Diagonal Random Chance
ax.plot([0, 1], [0, 1], color='#565f89', linestyle='--', label='Random Classifier (0.5000)')

ax.set_title("ROC Curve Comparison: Individual Gradient Boosters vs Blended Ensemble", color='white', fontsize=13, pad=15)
ax.set_xlabel("False Positive Rate (1 - Specificity)", color='#c0caf5', fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity)", color='#c0caf5', fontsize=11)
ax.grid(True, alpha=0.15)
ax.legend(facecolor='#24283b', edgecolor='#414868', labelcolor='white')

plt.tight_layout()
plt.show()


## 📦 6. Leaderboard Submission File Generation
We construct the final submission `submission.csv` containing blended probabilities.

In [ ]:
sub = pd.DataFrame({
    'id': test_raw['id'],
    'Will_Buy_EV': np.clip(test_blend, 1e-6, 1.0 - 1e-6)
})

sub.to_csv('submission.csv', index=False)
print(f"✅ Successfully exported submission.csv with {len(sub):,} predictions!")
print(f"Probability Bounds: [{sub['Will_Buy_EV'].min():.5f}, {sub['Will_Buy_EV'].max():.5f}]")
sub.head(6)
